# 300 · Functions & functional programming — procedural layer

**Mnemonic — rule of three: input → transform → output; the lambda.**

Codes in this notebook: **305, 314, 325, 333, 344, 356, 363, 373, 381, 394**

**Method:** for every code cell — read it, **PREDICT** the exact output, run it, **COMPARE**. A wrong prediction is the lesson: reread until the output feels inevitable. Run cells top to bottom in a single kernel. Exactly two cells are *supposed* to raise; both are marked `# INTENDED ERROR — read the traceback`.

## 305 · Default parameter values

A fallback value in the signature used when the caller omits that argument; Python evaluates it once, at definition time.

*A diner menu: "burger comes with fries unless you say otherwise" — but the kitchen keeps ONE shared basket of fries, reused for every order (the mutable-default trap).*

**Watch:** the second call to `bad_append` — does it get a fresh list, or last call's leftovers?

In [1]:
def bad_append(x, acc=[]):          # acc is created ONCE, at def time
    acc.append(x)
    return acc

print("call 1:", bad_append(1))
print("call 2:", bad_append(2))     # predict: [2]? the SAME basket of fries survives
print("the shared default itself:", bad_append.__defaults__)

call 1: [1]
call 2: [1, 2]
the shared default itself: ([1, 2],)


In [2]:
def good_append(x, acc=None):       # None sentinel: build the list INSIDE
    if acc is None:
        acc = []
    acc.append(x)
    return acc

print("call 1:", good_append(1))
print("call 2:", good_append(2))    # a fresh basket every order
print("default stays inert:", good_append.__defaults__)

call 1: [1]
call 2: [2]
default stays inert: (None,)


## 314 · Loop-variable capture bug

Closures created in a loop all share the single loop variable, so after the loop every one of them sees its final value.

*Ten balloons all tied to one weather vane: when the wind dies, all ten point wherever the vane last settled — nobody kept their own direction.*

**Watch:** three lambdas built while `i` was 0, 1, 2 — what does each return once the loop has ended?

In [3]:
fns = []
for i in range(3):
    fns.append(lambda: i)          # captures the VARIABLE i, not its value

print("loop finished, i =", i)     # the weather vane's final direction
print("buggy round:", [f() for f in fns])   # predict [0, 1, 2]?

loop finished, i = 2
buggy round: [2, 2, 2]


In [4]:
fns = []
for i in range(3):
    fns.append(lambda i=i: i)      # default arg evaluates i NOW: a private snapshot

print("fixed round:        ", [f() for f in fns])

comp = [lambda i=i: i for i in range(3)]    # same fix inside a comprehension
print("comprehension fixed:", [f() for f in comp])

fixed round:         [0, 1, 2]
comprehension fixed: [0, 1, 2]


## 325 · reduce (fold)

Collapses a sequence to a single value by repeatedly combining an accumulator with each element using a binary function.

*A snowball rolling downhill swallows flake after flake — each pass folds one more snowflake into the growing ball, and a single boulder arrives at the bottom.*

**Watch:** the shape shrinks stage by stage — list → list → one number. Predict all three printed values before running.

In [5]:
from functools import reduce

orders = [3, 7, 2, 9, 4, 11, 5]
print("input:        ", orders)

doubled = list(map(lambda x: x * 2, orders))
print("map (x2):     ", doubled)

big = list(filter(lambda x: x > 8, doubled))
print("filter (>8):  ", big)

total = reduce(lambda acc, x: acc + x, big, 0)   # the fold: one boulder arrives
print("reduce (sum): ", total)

input:         [3, 7, 2, 9, 4, 11, 5]
map (x2):      [6, 14, 4, 18, 8, 22, 10]
filter (>8):   [14, 18, 22, 10]
reduce (sum):  64


## 333 · Idempotence

Applying an operation more than once has the same effect as applying it once — `f(f(x)) == f(x)`; safe to retry.

*An elevator button already glowing: mash it five more times and the elevator's plan does not change. The first press did everything.*

**Watch:** which operations survive a double application unchanged — and which one compounds on every press?

In [6]:
print("abs once :", abs(-5))
print("abs twice:", abs(abs(-5)), "-> equal:", abs(abs(-5)) == abs(-5))

data = [3, 1, 2]
once = sorted(data)
twice = sorted(sorted(data))
print("sort once :", once)
print("sort twice:", twice, "-> equal:", once == twice)

s = {1, 2, 3}
print("s | s     :", sorted(s | s), "-> equal to s:", (s | s) == s)

abs once : 5
abs twice: 5 -> equal: True
sort once : [1, 2, 3]
sort twice: [1, 2, 3] -> equal: True
s | s     : [1, 2, 3] -> equal to s: True


In [7]:
counter = 0
counter += 1
print("after one press :", counter)
counter += 1
print("after two presses:", counter, "<- the effect compounds: NOT idempotent")
counter += 1
print("after three     :", counter, "<- every retry changes the world again")

after one press : 1
after two presses: 2 <- the effect compounds: NOT idempotent
after three     : 3 <- every retry changes the world again


## 344 · Defensive copy

Copy mutable data at trust boundaries — going into and out of your object — so outsiders holding a reference cannot mutate your internals.

*The museum hands visitors a replica sword to swing around while the real blade stays in the vault; scratch the copy all you like.*

**Watch:** the object's state after an outsider appends to what the getter returned — corrupted, or intact?

In [8]:
class Playlist:
    def __init__(self, tracks):
        self._tracks = tracks           # stores the caller's list directly
    def tracks(self):
        return self._tracks             # hands out the real blade

p = Playlist(["intro", "chorus"])
print("state before:", p.tracks())
stolen = p.tracks()
stolen.append("vandalism")              # outsider swings our internals around
print("state after :", p.tracks())

state before: ['intro', 'chorus']
state after : ['intro', 'chorus', 'vandalism']


In [9]:
class SafePlaylist:
    def __init__(self, tracks):
        self._tracks = list(tracks)     # copy IN at the boundary
    def tracks(self):
        return list(self._tracks)       # copy OUT: a replica sword

sp = SafePlaylist(["intro", "chorus"])
print("state before:", sp.tracks())
replica = sp.tracks()
replica.append("vandalism")             # only scratches the replica
print("state after :", sp.tracks())

state before: ['intro', 'chorus']
state after : ['intro', 'chorus']


## 356 · Currying (and partial application)

Transforming an n-argument function into a chain of one-argument functions: `f(a, b, c)` becomes `f(a)(b)(c)`. Partial application is the sibling: pre-fill some args of a normal function once.

*A curry house that accepts one spice at a time: hand over cumin and get back a chef who now wants coriander; only after the final spice does dinner appear.*

**Watch:** what `partial(pow, 2)` pre-fills (the base), and what `add(5)` returns — a value, or a chef waiting for the next argument?

In [10]:
from functools import partial

two_to_the = partial(pow, 2)            # pre-fill base=2 once
print("partial(pow, 2)(10):", two_to_the(10))
print("partial(pow, 2)(3): ", two_to_the(3))
print("partial(pow, 2)(0): ", two_to_the(0))

partial(pow, 2)(10): 1024
partial(pow, 2)(3):  8
partial(pow, 2)(0):  1


In [11]:
def add(a):                             # hand-rolled curry: one spice at a time
    def waiting_for_b(b):
        return a + b
    return waiting_for_b

add5 = add(5)
print("add(5) returns a:", type(add5).__name__, "- a chef waiting for b")
print("add(5)(3)  =", add5(3))
print("add(5)(40) =", add5(40))
print("add(1)(2)  =", add(1)(2))        # both spices in one breath

add(5) returns a: function - a chef waiting for b
add(5)(3)  = 8
add(5)(40) = 45
add(1)(2)  = 3


## 363 · Generator function

A function containing `yield`: calling it returns a generator, and each `next()` resumes the body until the next yield, pausing in place.

*A bard who freezes mid-sentence the instant he hands you a verse — statue-still, wine glass raised — and thaws exactly there when you tap his shoulder for the next line.*

**Watch:** the exact interleaving of prints inside vs. outside the body; what the third `next()` does; what a second pass over an exhausted generator yields.

In [12]:
def bard(n):
    print("  bard: clearing throat")            # runs only on FIRST next()
    for i in range(1, n + 1):
        print(f"  bard: about to yield verse {i}")
        yield f"verse {i}"
        print(f"  bard: thawed after verse {i}")
    print("  bard: bowing out")

g = bard(2)
print("generator created - predict: has anything printed from inside yet?")
print("got:", next(g))
print("got:", next(g))

generator created - predict: has anything printed from inside yet?
  bard: clearing throat
  bard: about to yield verse 1
got: verse 1
  bard: thawed after verse 1
  bard: about to yield verse 2
got: verse 2


In [13]:
# INTENDED ERROR — read the traceback
print("tapping the exhausted bard's shoulder one more time...")
next(g)   # body resumes, bows out, then raises StopIteration

tapping the exhausted bard's shoulder one more time...
  bard: thawed after verse 2
  bard: bowing out


StopIteration: 

In [14]:
squares = (x * x for x in range(4))
print("first pass :", [s for s in squares])
print("second pass:", [s for s in squares], "<- one-shot: already exhausted")

materialized = list(x * x for x in range(4))    # list() drains it ONCE, keeps all
print("list() copy:", materialized)
print("walk again :", [s for s in materialized], "<- a list can be re-walked forever")

first pass : [0, 1, 4, 9]
second pass: [] <- one-shot: already exhausted
list() copy: [0, 1, 4, 9]
walk again : [0, 1, 4, 9] <- a list can be re-walked forever


## 373 · Destructuring

Binding names to pieces of a structure in one pattern-shaped assignment — unpacking tuples, dicts, or objects by their shape.

*A chocolate orange: one sharp twist and it falls apart into labeled segments that land in your named, cupped fingers.*

**Watch:** which elements `*mid` swallows; which dict wins the merge on clashing keys; which `case` each shape falls into.

In [15]:
a, *mid, b = [1, 2, 3, 4, 5]        # star soaks up the middle segments
print("a  :", a)
print("mid:", mid)
print("b  :", b)

defaults = {"x": 0, "y": 0, "z": 0}
point = {"x": 3, "y": 4}
merged = {**defaults, **point}       # dict unpack: later keys win the clash
print("merged:", merged)

a  : 1
mid: [2, 3, 4]
b  : 5
merged: {'x': 3, 'y': 4, 'z': 0}


In [16]:
def describe(shape):
    match shape:
        case ("point", x, y):                # shape-driven binding + dispatch
            return f"point at x={x}, y={y}"
        case ("circle", r):
            return f"circle of radius {r}"
        case _:
            return "unknown shape"

print(describe(("point", 3, 4)))
print(describe(("circle", 7)))
print(describe(("blob",)))

point at x=3, y=4
circle of radius 7
unknown shape


## 381 · Option / Maybe

A container holding either one value (Some/Just) or nothing (None/Nothing), forcing callers to acknowledge absence in the type.

*A ring box that might be empty: the box's very shape forces you to open it and look before proposing — you cannot slide an unopened box onto a finger.*

**Watch:** the None-checking pyramid versus one helper that acknowledges absence exactly once — same two answers, radically different shape.

In [17]:
user  = {"profile": {"address": {"city": "Lisbon"}}}
ghost = {"profile": {}}                       # the address level is missing

def city_pyramid(d):                          # open every box by hand
    profile = d.get("profile")
    if profile is not None:
        address = profile.get("address")
        if address is not None:
            city = address.get("city")
            if city is not None:
                return city
    return "<missing>"

print("present:", city_pyramid(user))
print("missing:", city_pyramid(ghost))

present: Lisbon
missing: <missing>


In [18]:
def get_in(d, path, default=None):            # acknowledge absence ONCE
    for key in path:
        if not isinstance(d, dict) or key not in d:
            return default                    # the ring box was empty
        d = d[key]
    return d

print("present:", get_in(user,  ["profile", "address", "city"], "<missing>"))
print("missing:", get_in(ghost, ["profile", "address", "city"], "<missing>"))

present: Lisbon
missing: <missing>


## 394 · Duck typing

Dynamic-language polymorphism: any object with the right methods works, checked only at the call — if it quacks, it's a duck.

*A duck-call contest judged blindfolded: a robot, a comedian, and an actual duck all advance, because the judge only ever listens for the quack().*

**Watch:** the judge never checks types — only the quack. What happens the moment a Rock without one steps up?

In [19]:
class Duck:
    def quack(self):
        return "quack!"

class Robot:
    def quack(self):
        return "QUACK.EXE"

def judge(contestant):                      # blindfolded: no isinstance anywhere
    print("heard:", contestant.quack())

judge(Duck())
judge(Robot())

heard: quack!
heard: QUACK.EXE


In [20]:
# INTENDED ERROR — read the traceback
class Rock:
    pass                                    # no quack() at all

print("a rock rolls up to the microphone...")
judge(Rock())   # AttributeError, only discovered at the call

a rock rolls up to the microphone...


AttributeError: 'Rock' object has no attribute 'quack'